# F2: Transparent Pro Forma - Making Every Calculation Visible

---

## What This Notebook Does Differently

Most pro forma tools are **black boxes**. You put in assumptions, get out a "yes/no" answer, 
but can't see how the math works.

This notebook makes **every intermediate calculation explicit**:
- Each formula is shown in plain text before code
- Each cost component is broken down visibly
- Time value of money is computed step-by-step
- Policy levers are separated so you can see what changes what

Based on the [Terner Center "Making It Pencil" framework](https://ternercenter.berkeley.edu/research-and-policy/making-it-pencil-2023/), 
but with full transparency.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Explain what "making it pencil" means** in terms of investor return requirements
2. **Build a cost stack** from land through financing costs
3. **Model a construction draw schedule** and compute interest during construction
4. **Calculate key metrics**: NOI, Return on Cost, IRR, Equity Multiple
5. **Quantify policy impacts**: How parking, fees, and height rules affect feasibility
6. **Run scenario comparisons** to see what makes projects pencil or not

---

## Prerequisites

- Basic Python (variables, functions, dictionaries)
- Familiarity with spreadsheets (rows, columns, formulas)
- No prior real estate finance knowledge required

---

## 1. Introduction: What Does "Pencil" Mean?

### The Core Question

A project "pencils" when:

```
Expected Return  >=  Required Return (given the risk)
```

### Who Needs to Be Satisfied?

| Stakeholder | What They Provide | What They Require |
|-------------|-------------------|-------------------|
| **Construction Lender** | 60-70% of costs | Interest payments + full repayment |
| **Equity Investor** | 30-40% of costs | 15-20% IRR over hold period |
| **Permanent Lender** | Refinances construction loan | Debt coverage ratio > 1.25 |

If the project can't meet these requirements, **it doesn't get built**.

### Key Metrics We'll Calculate

| Metric | Formula | What It Tells You |
|--------|---------|-------------------|
| **Return on Cost (ROC)** | NOI / Total Development Cost | Return from building (vs. buying) |
| **Cap Rate** | NOI / Property Value | Return from owning |
| **IRR** | Rate where NPV = 0 | Total return over time |
| **Equity Multiple** | Total Distributions / Equity Invested | How many times you get your money back |
| **Debt Coverage Ratio** | NOI / Debt Service | Can you pay the mortgage? |

---

## 2. Setup and Imports

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
from pathlib import Path

# For cleaner number formatting
pd.options.display.float_format = '{:,.0f}'.format

# Constants
MONTHS_PER_YEAR = 12

print("Setup complete. All calculations will be shown step-by-step.")

## 3. Baseline Project Assumptions

We model a **120-unit podium apartment building** similar to Terner Center case studies.

### Project Physical Characteristics

| Attribute | Value | Notes |
|-----------|-------|-------|
| Site Size | 0.5 acres (21,780 SF) | Typical urban infill |
| Building Type | 5-story podium | Concrete parking + wood frame |
| Unit Count | 120 units | Mix of studios, 1BR, 2BR |
| Average Unit Size | 750 SF | Net rentable |
| Gross Building SF | 103,500 SF | Includes corridors, amenities |
| Parking Spaces | 120 (1.0/unit) | Structured, below grade |

### Why These Numbers Matter

- **Podium construction** (concrete base + wood frame above) is common for 4-6 story buildings
- **1.0 parking ratio** is typical for suburban/auto-oriented areas; urban areas may be 0.5 or less
- **750 SF average** reflects market preference for smaller, more affordable units

In [ ]:
@dataclass
class ProjectConfig:
    """
    All project assumptions in one place.
    
    This is the SINGLE SOURCE OF TRUTH for the pro forma.
    Change values here to run different scenarios.
    """
    
    # ============ PHYSICAL CHARACTERISTICS ============
    site_sf: int = 21_780                    # Site size in square feet (0.5 acres)
    num_units: int = 120                     # Total residential units
    avg_unit_sf: int = 750                   # Average unit size (net rentable)
    efficiency_ratio: float = 0.87           # Net rentable / Gross SF
    num_floors: int = 5                      # Building height
    
    # Unit mix (as shares)
    unit_mix: Dict[str, float] = field(default_factory=lambda: {
        'studio': 0.20,   # 20% studios
        '1br': 0.50,      # 50% one-bedrooms
        '2br': 0.30       # 30% two-bedrooms
    })
    
    # ============ POLICY-SENSITIVE INPUTS ============
    # These are the levers that policy changes can pull
    
    parking_ratio: float = 1.0               # Spaces per unit
    impact_fee_per_unit: float = 35_000      # City impact fees
    inclusionary_pct: float = 0.0            # % of units at below-market rent
    
    # Height/seismic/fire cost increments (per SF above threshold)
    height_threshold_floors: int = 3         # Type V wood limit
    height_cost_premium_per_sf: float = 50   # $/SF for Type III or I-A construction
    
    # ============ HARD COSTS ============
    hard_cost_per_sf: float = 400            # Base construction cost $/SF
    parking_cost_per_space: float = 55_000   # Structured parking $/space
    site_work_per_sf: float = 15             # Site prep, utilities $/SF site
    contingency_pct: float = 0.05            # 5% of hard costs
    
    # ============ SOFT COSTS ============
    architecture_pct: float = 0.06           # % of hard costs
    engineering_pct: float = 0.02            # % of hard costs
    permits_and_fees_pct: float = 0.03       # % of hard costs (excluding impact fees)
    legal_and_accounting: float = 150_000    # Flat fee
    developer_fee_pct: float = 0.04          # % of total development cost
    
    # ============ LAND ============
    land_cost_per_sf: float = 350            # $/SF of land
    acquisition_costs_pct: float = 0.02      # Closing costs as % of land
    
    # ============ FINANCING ============
    construction_ltc: float = 0.65           # Loan-to-cost ratio
    construction_rate: float = 0.08          # Annual interest rate
    construction_fee_pct: float = 0.01       # Origination fee
    
    permanent_ltv: float = 0.65              # Loan-to-value for perm loan
    permanent_rate: float = 0.065            # Permanent loan rate
    permanent_term_years: int = 30           # Amortization period
    
    # ============ TIMELINE ============
    predevelopment_months: int = 12          # Entitlements, design
    construction_months: int = 24            # Construction period
    lease_up_months: int = 12                # Time to reach stabilization
    hold_period_years: int = 7               # Total hold before sale/refi
    
    # ============ OPERATING ASSUMPTIONS ============
    rent_per_sf: Dict[str, float] = field(default_factory=lambda: {
        'studio': 4.00,   # $/SF/month
        '1br': 3.75,
        '2br': 3.50
    })
    unit_sizes: Dict[str, int] = field(default_factory=lambda: {
        'studio': 500,
        '1br': 700,
        '2br': 1000
    })
    vacancy_rate: float = 0.05               # Stabilized vacancy
    operating_expense_ratio: float = 0.35    # OpEx as % of EGI
    rent_growth_rate: float = 0.03           # Annual rent growth
    expense_growth_rate: float = 0.025       # Annual expense growth
    
    # ============ EXIT ASSUMPTIONS ============
    exit_cap_rate: float = 0.05              # Cap rate at sale
    sales_cost_pct: float = 0.02             # Broker fees, closing costs
    
    # ============ INVESTOR REQUIREMENTS ============
    target_irr: float = 0.15                 # 15% levered IRR
    target_roc: float = 0.055                # 5.5% unlevered yield
    target_equity_multiple: float = 2.0     # 2x equity multiple
    
    @property
    def gross_building_sf(self) -> float:
        """Total building square footage including common areas."""
        return self.num_units * self.avg_unit_sf / self.efficiency_ratio
    
    @property
    def parking_spaces(self) -> int:
        """Total parking spaces required."""
        return int(self.num_units * self.parking_ratio)


# Create baseline project
project = ProjectConfig()

print("BASELINE PROJECT CONFIGURATION")
print("=" * 50)
print(f"Units: {project.num_units}")
print(f"Gross Building SF: {project.gross_building_sf:,.0f}")
print(f"Parking Spaces: {project.parking_spaces}")
print(f"Site SF: {project.site_sf:,}")
print()
print("POLICY-SENSITIVE INPUTS:")
print(f"  Parking Ratio: {project.parking_ratio} spaces/unit")
print(f"  Impact Fees: ${project.impact_fee_per_unit:,.0f}/unit")
print(f"  Inclusionary: {project.inclusionary_pct*100:.0f}%")

## 4. Cost Stack: Building Total Development Cost (TDC)

Total Development Cost is the sum of all costs to deliver a stabilized building:

```
TDC = Land Cost + Hard Costs + Soft Costs + Financing Costs + Fees
```

Let's build each component with **explicit formulas**.

---

### 4.1 Land Costs

**Formula:**
```
Land Cost = Site SF × Land Cost per SF
Acquisition Costs = Land Cost × Acquisition Cost %
Total Land = Land Cost + Acquisition Costs
```

In [ ]:
def compute_land_costs(p: ProjectConfig) -> pd.Series:
    """
    Calculate land acquisition costs.
    
    Returns a Series with each component labeled.
    """
    land_cost = p.site_sf * p.land_cost_per_sf
    acquisition_costs = land_cost * p.acquisition_costs_pct
    total = land_cost + acquisition_costs
    
    return pd.Series({
        'Land Purchase': land_cost,
        'Acquisition Costs (closing, due diligence)': acquisition_costs,
        'TOTAL LAND COSTS': total
    })


# Calculate and display
land_costs = compute_land_costs(project)

print("LAND COSTS")
print("=" * 50)
print(f"Site: {project.site_sf:,} SF × ${project.land_cost_per_sf:,.0f}/SF")
print()
for label, value in land_costs.items():
    print(f"{label:45} ${value:>12,.0f}")

### 4.2 Hard Costs

Hard costs are the physical construction expenses.

**Formulas:**
```
Base Construction = Gross Building SF × Hard Cost per SF
Height Premium = (Floors above threshold) × Building SF × Height Premium per SF
Parking = Parking Spaces × Cost per Space
Site Work = Site SF × Site Work per SF
Contingency = (Base + Height + Parking + Site) × Contingency %
```

In [ ]:
def compute_hard_costs(p: ProjectConfig) -> pd.Series:
    """
    Calculate hard (construction) costs.
    
    Includes height premium for buildings above Type V wood threshold.
    """
    gross_sf = p.gross_building_sf
    
    # Base construction
    base_construction = gross_sf * p.hard_cost_per_sf
    
    # Height premium (for floors above threshold)
    floors_above_threshold = max(0, p.num_floors - p.height_threshold_floors)
    height_premium = floors_above_threshold * gross_sf * p.height_cost_premium_per_sf / p.num_floors
    
    # Parking (structured)
    parking_cost = p.parking_spaces * p.parking_cost_per_space
    
    # Site work
    site_work = p.site_sf * p.site_work_per_sf
    
    # Subtotal before contingency
    subtotal = base_construction + height_premium + parking_cost + site_work
    
    # Contingency
    contingency = subtotal * p.contingency_pct
    
    total = subtotal + contingency
    
    return pd.Series({
        'Base Construction': base_construction,
        f'Height Premium ({floors_above_threshold} floors above Type V)': height_premium,
        f'Structured Parking ({p.parking_spaces} spaces)': parking_cost,
        'Site Work': site_work,
        f'Contingency ({p.contingency_pct*100:.0f}%)': contingency,
        'TOTAL HARD COSTS': total
    })


# Calculate and display
hard_costs = compute_hard_costs(project)

print("HARD COSTS")
print("=" * 50)
print(f"Building: {project.gross_building_sf:,.0f} SF × ${project.hard_cost_per_sf:,.0f}/SF")
print()
for label, value in hard_costs.items():
    print(f"{label:50} ${value:>12,.0f}")

print()
print(f"Hard Cost per Unit: ${hard_costs['TOTAL HARD COSTS'] / project.num_units:,.0f}")
print(f"Hard Cost per SF: ${hard_costs['TOTAL HARD COSTS'] / project.gross_building_sf:,.0f}")

### 4.3 Soft Costs

Soft costs are everything that isn't physical construction.

**Formulas:**
```
Architecture = Hard Costs × Architecture %
Engineering = Hard Costs × Engineering %
Permits = Hard Costs × Permits %
Impact Fees = Units × Impact Fee per Unit
Developer Fee = TDC × Developer Fee %  (circular, often estimated)
```

In [ ]:
def compute_soft_costs(p: ProjectConfig, hard_costs_total: float) -> pd.Series:
    """
    Calculate soft costs.
    
    Note: Developer fee is circular (based on TDC which includes dev fee).
    We estimate TDC as hard + soft + land + ~10% financing.
    """
    # Professional fees (as % of hard costs)
    architecture = hard_costs_total * p.architecture_pct
    engineering = hard_costs_total * p.engineering_pct
    permits_fees = hard_costs_total * p.permits_and_fees_pct
    
    # Impact fees (per unit)
    impact_fees = p.num_units * p.impact_fee_per_unit
    
    # Fixed costs
    legal_accounting = p.legal_and_accounting
    
    # Developer fee (estimate based on projected TDC)
    # TDC ≈ Hard + Soft + Land + Financing
    land_estimate = p.site_sf * p.land_cost_per_sf * 1.02
    soft_before_dev_fee = architecture + engineering + permits_fees + impact_fees + legal_accounting
    financing_estimate = hard_costs_total * 0.10  # rough estimate
    tdc_estimate = hard_costs_total + soft_before_dev_fee + land_estimate + financing_estimate
    developer_fee = tdc_estimate * p.developer_fee_pct
    
    total = architecture + engineering + permits_fees + impact_fees + legal_accounting + developer_fee
    
    return pd.Series({
        f'Architecture ({p.architecture_pct*100:.0f}% of hard)': architecture,
        f'Engineering ({p.engineering_pct*100:.0f}% of hard)': engineering,
        f'Permits & Fees ({p.permits_and_fees_pct*100:.0f}% of hard)': permits_fees,
        f'Impact Fees ({p.num_units} units × ${p.impact_fee_per_unit:,.0f})': impact_fees,
        'Legal & Accounting': legal_accounting,
        f'Developer Fee ({p.developer_fee_pct*100:.0f}% of TDC est.)': developer_fee,
        'TOTAL SOFT COSTS': total
    })


# Calculate and display
soft_costs = compute_soft_costs(project, hard_costs['TOTAL HARD COSTS'])

print("SOFT COSTS")
print("=" * 50)
for label, value in soft_costs.items():
    print(f"{label:55} ${value:>12,.0f}")

### 4.4 Financing Costs

Financing costs include:
- Loan origination fees
- Interest during construction (calculated on draw schedule)
- Interest reserve

**Key Concept: Draw Schedule**

Construction loans aren't drawn all at once. Money is drawn as construction progresses.
Interest accrues only on the drawn balance.

We'll use a simple **S-curve** draw schedule:
- Slow start (foundation, permits)
- Fast middle (vertical construction)
- Slow finish (finishes, punch list)

In [ ]:
def create_draw_schedule(total_costs: float, months: int) -> pd.DataFrame:
    """
    Create an S-curve construction draw schedule.
    
    Returns DataFrame with monthly draws and cumulative balance.
    """
    # S-curve: slow-fast-slow pattern
    # Using a sigmoid-like distribution
    t = np.linspace(-3, 3, months)
    s_curve = 1 / (1 + np.exp(-t))
    s_curve = (s_curve - s_curve.min()) / (s_curve.max() - s_curve.min())
    
    # Convert to monthly draws (first derivative of cumulative)
    cumulative_pct = s_curve
    monthly_pct = np.diff(cumulative_pct, prepend=0)
    
    # Normalize to sum to 100%
    monthly_pct = monthly_pct / monthly_pct.sum()
    
    df = pd.DataFrame({
        'Month': range(1, months + 1),
        'Draw %': monthly_pct,
        'Draw Amount': monthly_pct * total_costs,
        'Cumulative Drawn': np.cumsum(monthly_pct) * total_costs
    })
    
    return df


def compute_interest_during_construction(
    draw_schedule: pd.DataFrame, 
    annual_rate: float,
    ltc: float
) -> Tuple[float, pd.DataFrame]:
    """
    Calculate interest accrued during construction.
    
    Interest accrues on the outstanding loan balance each month.
    Loan balance = Cumulative Drawn × LTC (lender doesn't fund 100%)
    """
    monthly_rate = annual_rate / 12
    
    df = draw_schedule.copy()
    df['Loan Balance'] = df['Cumulative Drawn'] * ltc
    df['Monthly Interest'] = df['Loan Balance'] * monthly_rate
    df['Cumulative Interest'] = df['Monthly Interest'].cumsum()
    
    total_interest = df['Monthly Interest'].sum()
    
    return total_interest, df


# Calculate draw schedule
total_hard_soft = hard_costs['TOTAL HARD COSTS'] + soft_costs['TOTAL SOFT COSTS']
draw_schedule = create_draw_schedule(total_hard_soft, project.construction_months)

# Calculate interest during construction
interest_during_construction, schedule_with_interest = compute_interest_during_construction(
    draw_schedule, 
    project.construction_rate,
    project.construction_ltc
)

print("CONSTRUCTION DRAW SCHEDULE (showing every 3 months)")
print("=" * 80)
display_schedule = schedule_with_interest[schedule_with_interest['Month'] % 3 == 0].copy()
display_schedule['Draw %'] = display_schedule['Draw %'].apply(lambda x: f"{x*100:.1f}%")
print(display_schedule.to_string(index=False))

print()
print(f"Total Interest During Construction: ${interest_during_construction:,.0f}")
print(f"Average Outstanding Balance: ${schedule_with_interest['Loan Balance'].mean():,.0f}")

In [ ]:
def compute_financing_costs(p: ProjectConfig, hard_soft_total: float) -> pd.Series:
    """
    Calculate all financing-related costs.
    """
    # Create draw schedule
    draw = create_draw_schedule(hard_soft_total, p.construction_months)
    
    # Interest during construction
    interest, _ = compute_interest_during_construction(
        draw, p.construction_rate, p.construction_ltc
    )
    
    # Loan amount and fees
    loan_amount = hard_soft_total * p.construction_ltc
    origination_fee = loan_amount * p.construction_fee_pct
    
    # Interest reserve (for lease-up period)
    # Estimate average balance at 90% of peak
    interest_reserve = loan_amount * 0.9 * (p.construction_rate / 12) * p.lease_up_months
    
    total = interest + origination_fee + interest_reserve
    
    return pd.Series({
        f'Interest During Construction ({p.construction_months} mo)': interest,
        f'Loan Origination Fee ({p.construction_fee_pct*100:.0f}%)': origination_fee,
        f'Interest Reserve ({p.lease_up_months} mo lease-up)': interest_reserve,
        'TOTAL FINANCING COSTS': total
    })


# Calculate financing costs
financing_costs = compute_financing_costs(project, total_hard_soft)

print("FINANCING COSTS")
print("=" * 50)
for label, value in financing_costs.items():
    print(f"{label:55} ${value:>12,.0f}")

### 4.5 Total Development Cost Summary

Now we assemble all cost components into the **Total Development Cost (TDC)**.

In [ ]:
def compute_total_development_cost(p: ProjectConfig) -> pd.DataFrame:
    """
    Calculate complete Total Development Cost breakdown.
    
    Returns DataFrame with each category and percentage of total.
    """
    # Calculate each component
    land = compute_land_costs(p)
    hard = compute_hard_costs(p)
    
    hard_total = hard['TOTAL HARD COSTS']
    soft = compute_soft_costs(p, hard_total)
    
    hard_soft = hard_total + soft['TOTAL SOFT COSTS']
    financing = compute_financing_costs(p, hard_soft)
    
    # Assemble summary
    categories = {
        'Land': land['TOTAL LAND COSTS'],
        'Hard Costs': hard_total,
        'Soft Costs': soft['TOTAL SOFT COSTS'],
        'Financing Costs': financing['TOTAL FINANCING COSTS']
    }
    
    total = sum(categories.values())
    
    df = pd.DataFrame([
        {'Category': k, 'Amount': v, '% of TDC': v/total*100, 'Per Unit': v/p.num_units}
        for k, v in categories.items()
    ])
    
    # Add total row
    df = pd.concat([df, pd.DataFrame([{
        'Category': 'TOTAL DEVELOPMENT COST',
        'Amount': total,
        '% of TDC': 100.0,
        'Per Unit': total / p.num_units
    }])], ignore_index=True)
    
    return df, {'land': land, 'hard': hard, 'soft': soft, 'financing': financing, 'total': total}


# Calculate TDC
tdc_summary, cost_details = compute_total_development_cost(project)

print("TOTAL DEVELOPMENT COST SUMMARY")
print("=" * 70)
print(f"Project: {project.num_units} units, {project.gross_building_sf:,.0f} SF")
print()

for _, row in tdc_summary.iterrows():
    if row['Category'] == 'TOTAL DEVELOPMENT COST':
        print("-" * 70)
    print(f"{row['Category']:25} ${row['Amount']:>14,.0f}  ({row['% of TDC']:>5.1f}%)  ${row['Per Unit']:>10,.0f}/unit")

print()
print(f"Cost per SF: ${cost_details['total'] / project.gross_building_sf:,.0f}")

## 5. Operating Cash Flow and NOI

Now we calculate what the building earns once it's built and leased.

### Net Operating Income Formula

```
Gross Potential Rent (GPR) = Sum of (Units × Rent per Unit) for each unit type
Less: Vacancy = GPR × Vacancy Rate
= Effective Gross Income (EGI)
Less: Operating Expenses = EGI × OpEx Ratio
= Net Operating Income (NOI)
```

In [ ]:
def compute_rent_roll(p: ProjectConfig) -> pd.DataFrame:
    """
    Calculate detailed rent roll by unit type.
    """
    rows = []
    
    for unit_type, share in p.unit_mix.items():
        count = int(p.num_units * share)
        size = p.unit_sizes[unit_type]
        rent_per_sf = p.rent_per_sf[unit_type]
        monthly_rent = size * rent_per_sf
        annual_rent = monthly_rent * 12 * count
        
        rows.append({
            'Unit Type': unit_type.upper(),
            'Count': count,
            'Size (SF)': size,
            'Rent/SF': rent_per_sf,
            'Monthly Rent': monthly_rent,
            'Annual GPR': annual_rent
        })
    
    df = pd.DataFrame(rows)
    
    # Add totals
    totals = pd.DataFrame([{
        'Unit Type': 'TOTAL',
        'Count': df['Count'].sum(),
        'Size (SF)': '',
        'Rent/SF': '',
        'Monthly Rent': '',
        'Annual GPR': df['Annual GPR'].sum()
    }])
    
    return pd.concat([df, totals], ignore_index=True)


def compute_noi(p: ProjectConfig) -> pd.Series:
    """
    Calculate Net Operating Income step-by-step.
    """
    # Gross Potential Rent
    rent_roll = compute_rent_roll(p)
    gpr = rent_roll[rent_roll['Unit Type'] == 'TOTAL']['Annual GPR'].values[0]
    
    # Vacancy
    vacancy = gpr * p.vacancy_rate
    egi = gpr - vacancy
    
    # Operating expenses
    opex = egi * p.operating_expense_ratio
    
    # NOI
    noi = egi - opex
    
    return pd.Series({
        'Gross Potential Rent': gpr,
        f'Less: Vacancy ({p.vacancy_rate*100:.0f}%)': -vacancy,
        'Effective Gross Income': egi,
        f'Less: Operating Expenses ({p.operating_expense_ratio*100:.0f}%)': -opex,
        'NET OPERATING INCOME': noi
    })


# Calculate rent roll
rent_roll = compute_rent_roll(project)

print("RENT ROLL")
print("=" * 70)
print(rent_roll.to_string(index=False))

print()

# Calculate NOI
noi_breakdown = compute_noi(project)

print("NET OPERATING INCOME CALCULATION")
print("=" * 50)
for label, value in noi_breakdown.items():
    if value < 0:
        print(f"{label:45} (${-value:>11,.0f})")
    else:
        print(f"{label:45} ${value:>12,.0f}")

## 6. Key Metrics: ROC, IRR, and Equity Multiple

Now we calculate the metrics that determine if the project "pencils."

### Return on Cost (ROC)

**Formula:**
```
Return on Cost = Stabilized NOI / Total Development Cost
```

**Interpretation:**
- ROC > Exit Cap Rate + ~100 bps = Project creates value
- ROC < Exit Cap Rate = Cheaper to buy existing building

In [ ]:
# Calculate Return on Cost
stabilized_noi = noi_breakdown['NET OPERATING INCOME']
total_dev_cost = cost_details['total']

roc = stabilized_noi / total_dev_cost

print("RETURN ON COST (ROC)")
print("=" * 50)
print(f"Stabilized NOI:         ${stabilized_noi:>15,.0f}")
print(f"Total Development Cost: ${total_dev_cost:>15,.0f}")
print(f"")
print(f"Return on Cost:         {roc*100:>14.2f}%")
print(f"Exit Cap Rate:          {project.exit_cap_rate*100:>14.2f}%")
print(f"Spread (ROC - Cap):     {(roc - project.exit_cap_rate)*100:>14.2f}%")
print()

if roc >= project.target_roc:
    print(f"ROC meets target of {project.target_roc*100:.1f}%. Project MAY pencil.")
else:
    print(f"ROC below target of {project.target_roc*100:.1f}%. Project DOES NOT pencil on unlevered basis.")

### Internal Rate of Return (IRR)

IRR is the discount rate that makes the Net Present Value of all cash flows equal to zero.

**Formula:**
```
NPV = Σ (Cash Flow_t / (1 + IRR)^t) = 0
```

We need to build a period-by-period cash flow model:

| Period | Cash Flow Components |
|--------|---------------------|
| Year 0 | - Equity contribution |
| Years 1-2 | - More equity (construction) |
| Year 3+ | + NOI - Debt Service |
| Final Year | + Sale Proceeds - Loan Payoff |

In [ ]:
def compute_irr_and_cash_flows(p: ProjectConfig, cost_details: dict) -> Tuple[float, pd.DataFrame]:
    """
    Build year-by-year cash flow model and compute IRR.
    
    Returns IRR and detailed cash flow DataFrame.
    """
    tdc = cost_details['total']
    noi_year1 = compute_noi(p)['NET OPERATING INCOME']
    
    # Financing structure
    construction_loan = tdc * p.construction_ltc
    equity_required = tdc - construction_loan
    
    # Permanent loan (at stabilization)
    stabilized_value = noi_year1 / p.exit_cap_rate
    perm_loan = min(stabilized_value * p.permanent_ltv, construction_loan)  # refinance
    
    # Annual debt service on permanent loan
    # P&I payment = Loan × [r(1+r)^n] / [(1+r)^n - 1]
    r = p.permanent_rate / 12
    n = p.permanent_term_years * 12
    monthly_payment = perm_loan * (r * (1+r)**n) / ((1+r)**n - 1)
    annual_debt_service = monthly_payment * 12
    
    # Build cash flows
    total_years = p.hold_period_years
    construction_years = (p.predevelopment_months + p.construction_months + p.lease_up_months) / 12
    
    cash_flows = []
    
    # Year 0: Initial equity
    cash_flows.append({
        'Year': 0,
        'NOI': 0,
        'Debt Service': 0,
        'Equity Investment': -equity_required * 0.3,  # 30% upfront
        'Sale Proceeds': 0,
        'Net Cash Flow': -equity_required * 0.3
    })
    
    # Year 1: Construction equity
    cash_flows.append({
        'Year': 1,
        'NOI': 0,
        'Debt Service': 0,
        'Equity Investment': -equity_required * 0.5,  # 50% during construction
        'Sale Proceeds': 0,
        'Net Cash Flow': -equity_required * 0.5
    })
    
    # Year 2: Final construction + partial NOI
    cash_flows.append({
        'Year': 2,
        'NOI': noi_year1 * 0.25,  # partial year operations
        'Debt Service': -annual_debt_service * 0.5,
        'Equity Investment': -equity_required * 0.2,  # remaining 20%
        'Sale Proceeds': 0,
        'Net Cash Flow': noi_year1 * 0.25 - annual_debt_service * 0.5 - equity_required * 0.2
    })
    
    # Years 3 to N-1: Stabilized operations
    noi = noi_year1
    for year in range(3, total_years):
        noi = noi * (1 + p.rent_growth_rate)  # rent growth
        cash_flow_from_ops = noi - annual_debt_service
        
        cash_flows.append({
            'Year': year,
            'NOI': noi,
            'Debt Service': -annual_debt_service,
            'Equity Investment': 0,
            'Sale Proceeds': 0,
            'Net Cash Flow': cash_flow_from_ops
        })
    
    # Final year: Sale
    final_noi = noi * (1 + p.rent_growth_rate)
    sale_price = final_noi / p.exit_cap_rate
    sales_costs = sale_price * p.sales_cost_pct
    
    # Loan payoff (simplified - assume interest-only for now)
    loan_payoff = perm_loan * 0.95  # ~5% principal paydown over hold
    
    net_sale_proceeds = sale_price - sales_costs - loan_payoff
    
    cash_flows.append({
        'Year': total_years,
        'NOI': final_noi,
        'Debt Service': -annual_debt_service,
        'Equity Investment': 0,
        'Sale Proceeds': net_sale_proceeds,
        'Net Cash Flow': final_noi - annual_debt_service + net_sale_proceeds
    })
    
    df = pd.DataFrame(cash_flows)
    
    # Calculate IRR
    cf_array = df['Net Cash Flow'].values
    irr = np.irr(cf_array)
    
    # Equity multiple
    total_distributions = df[df['Net Cash Flow'] > 0]['Net Cash Flow'].sum()
    total_invested = -df[df['Equity Investment'] < 0]['Equity Investment'].sum()
    equity_multiple = (total_distributions + total_invested) / total_invested
    
    return irr, equity_multiple, df


# Calculate IRR
irr, equity_multiple, cash_flow_df = compute_irr_and_cash_flows(project, cost_details)

print("CASH FLOW PROJECTION")
print("=" * 90)
print(cash_flow_df.to_string(index=False))

print()
print("=" * 50)
print("INVESTOR RETURNS")
print("=" * 50)
print(f"Internal Rate of Return (IRR):  {irr*100:>10.2f}%")
print(f"Equity Multiple:                {equity_multiple:>10.2f}x")
print(f"")
print(f"Target IRR:                     {project.target_irr*100:>10.2f}%")
print(f"Target Equity Multiple:         {project.target_equity_multiple:>10.2f}x")
print()

if irr >= project.target_irr:
    print("PROJECT PENCILS on levered basis.")
else:
    print(f"PROJECT DOES NOT PENCIL. IRR is {(project.target_irr - irr)*100:.1f}% below target.")

## 7. Policy Lever Analysis

Now let's see how policy choices affect feasibility.

We'll analyze three key levers:
1. **Parking requirements** - Structured parking costs $50K+ per space
2. **Impact fees** - Berkeley charges $35K+ per unit
3. **Height/construction type** - Type I (concrete) costs more than Type V (wood)

In [ ]:
def run_scenario(params: dict) -> dict:
    """
    Run a complete pro forma scenario with given parameters.
    
    Args:
        params: Dictionary of parameters to override from baseline
    
    Returns:
        Dictionary with all key results
    """
    # Create project config with overrides
    p = ProjectConfig(**params)
    
    # Calculate costs
    tdc_df, cost_details = compute_total_development_cost(p)
    tdc = cost_details['total']
    
    # Calculate NOI
    noi_series = compute_noi(p)
    noi = noi_series['NET OPERATING INCOME']
    
    # Calculate metrics
    roc = noi / tdc
    
    try:
        irr, equity_multiple, cf_df = compute_irr_and_cash_flows(p, cost_details)
    except:
        irr = 0
        equity_multiple = 0
    
    return {
        'TDC': tdc,
        'TDC_per_unit': tdc / p.num_units,
        'NOI': noi,
        'ROC': roc,
        'IRR': irr,
        'Equity_Multiple': equity_multiple,
        'Pencils_ROC': roc >= p.target_roc,
        'Pencils_IRR': irr >= p.target_irr,
        'config': p
    }


# Define scenarios
scenarios = {
    'Baseline': {},
    
    'No Parking': {
        'parking_ratio': 0.0
    },
    
    'Reduced Parking (0.5/unit)': {
        'parking_ratio': 0.5
    },
    
    'Lower Fees ($10K/unit)': {
        'impact_fee_per_unit': 10_000
    },
    
    'No Fees': {
        'impact_fee_per_unit': 0
    },
    
    'Higher Rents (+$0.50/SF)': {
        'rent_per_sf': {'studio': 4.50, '1br': 4.25, '2br': 4.00}
    },
    
    'Lower Construction (-10%)': {
        'hard_cost_per_sf': 360
    },
    
    'Combined Reforms': {
        'parking_ratio': 0.5,
        'impact_fee_per_unit': 15_000,
        'hard_cost_per_sf': 380
    },
    
    'All Levers Pulled': {
        'parking_ratio': 0.25,
        'impact_fee_per_unit': 10_000,
        'hard_cost_per_sf': 360,
        'rent_per_sf': {'studio': 4.25, '1br': 4.00, '2br': 3.75}
    }
}

# Run all scenarios
results = []
for name, params in scenarios.items():
    result = run_scenario(params)
    results.append({
        'Scenario': name,
        'TDC ($M)': result['TDC'] / 1_000_000,
        'Cost/Unit ($K)': result['TDC_per_unit'] / 1_000,
        'NOI ($M)': result['NOI'] / 1_000_000,
        'ROC': f"{result['ROC']*100:.2f}%",
        'IRR': f"{result['IRR']*100:.1f}%",
        'Eq Mult': f"{result['Equity_Multiple']:.2f}x",
        'Pencils?': 'YES' if result['Pencils_IRR'] else 'NO'
    })

results_df = pd.DataFrame(results)

print("SCENARIO COMPARISON")
print("=" * 100)
print(f"Target IRR: {project.target_irr*100:.0f}%   Target ROC: {project.target_roc*100:.1f}%")
print()
print(results_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

# Visualize scenario impacts
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC comparison
ax1 = axes[0]
roc_values = [float(r['ROC'].strip('%')) for r in results]
colors = ['green' if v >= project.target_roc*100 else 'red' for v in roc_values]
bars = ax1.barh(results_df['Scenario'], roc_values, color=colors, alpha=0.7)
ax1.axvline(x=project.target_roc*100, color='black', linestyle='--', label=f'Target ({project.target_roc*100:.1f}%)')
ax1.set_xlabel('Return on Cost (%)')
ax1.set_title('Return on Cost by Scenario')
ax1.legend()

# IRR comparison
ax2 = axes[1]
irr_values = [float(r['IRR'].strip('%')) for r in results]
colors = ['green' if v >= project.target_irr*100 else 'red' for v in irr_values]
bars = ax2.barh(results_df['Scenario'], irr_values, color=colors, alpha=0.7)
ax2.axvline(x=project.target_irr*100, color='black', linestyle='--', label=f'Target ({project.target_irr*100:.0f}%)')
ax2.set_xlabel('Internal Rate of Return (%)')
ax2.set_title('IRR by Scenario')
ax2.legend()

plt.tight_layout()
plt.show()

## 8. Sensitivity Analysis

Which inputs have the biggest impact on feasibility?

In [ ]:
def sensitivity_analysis():
    """
    Test sensitivity of IRR to key inputs.
    """
    baseline = run_scenario({})
    baseline_irr = baseline['IRR']
    
    sensitivities = []
    
    # Test each variable at +/- 20%
    tests = [
        ('Hard Cost/SF', 'hard_cost_per_sf', 400, 0.20),
        ('Land Cost/SF', 'land_cost_per_sf', 350, 0.20),
        ('Impact Fees/Unit', 'impact_fee_per_unit', 35_000, 0.20),
        ('Parking Ratio', 'parking_ratio', 1.0, 0.50),  # 50% change
        ('Construction Rate', 'construction_rate', 0.08, 0.25),
        ('Exit Cap Rate', 'exit_cap_rate', 0.05, 0.20),
    ]
    
    for label, param, base_val, pct_change in tests:
        # Test higher value
        high_val = base_val * (1 + pct_change)
        high_result = run_scenario({param: high_val})
        
        # Test lower value
        low_val = base_val * (1 - pct_change)
        low_result = run_scenario({param: low_val})
        
        # Calculate sensitivity
        irr_range = high_result['IRR'] - low_result['IRR']
        
        sensitivities.append({
            'Variable': label,
            f'Low ({-pct_change*100:.0f}%)': f"{low_result['IRR']*100:.1f}%",
            'Baseline': f"{baseline_irr*100:.1f}%",
            f'High (+{pct_change*100:.0f}%)': f"{high_result['IRR']*100:.1f}%",
            'IRR Swing': f"{abs(irr_range)*100:.1f}%"
        })
    
    # Sort by impact
    sensitivities.sort(key=lambda x: float(x['IRR Swing'].strip('%')), reverse=True)
    
    return pd.DataFrame(sensitivities)


sens_df = sensitivity_analysis()

print("SENSITIVITY ANALYSIS: What Moves IRR Most?")
print("=" * 80)
print(sens_df.to_string(index=False))
print()
print("Interpretation: Variables with larger IRR Swing have greater impact on feasibility.")

## 9. Summary and Key Formulas

### Formulas Implemented

| Metric | Formula |
|--------|--------|
| **TDC** | Land + Hard Costs + Soft Costs + Financing |
| **Hard Costs** | (Building SF × $/SF) + (Parking × $/space) + Site Work + Contingency |
| **Interest During Construction** | Σ (Monthly Balance × Monthly Rate) |
| **NOI** | GPR - Vacancy - Operating Expenses |
| **ROC** | NOI / TDC |
| **IRR** | Rate where NPV of all cash flows = 0 |
| **Debt Service** | Loan × [r(1+r)^n / ((1+r)^n - 1)] × 12 |

### Key Findings from This Analysis

1. **Baseline project does not pencil** at current costs and rents
2. **Parking is a major cost driver** - eliminating it can add 1-2% to IRR
3. **Impact fees matter** - reducing from $35K to $10K improves feasibility
4. **No single lever is sufficient** - need to stack multiple reforms
5. **Exit cap rate is highly sensitive** - small changes have big IRR impacts

### How to Use This Notebook

1. **Modify `ProjectConfig`** at the top to test your own assumptions
2. **Add scenarios** to the `scenarios` dictionary
3. **Run `run_scenario(params)`** for any parameter combination
4. **Export results** to CSV for further analysis

---

## Resources

- [Terner Center: Making It Pencil (2023)](https://ternercenter.berkeley.edu/research-and-policy/making-it-pencil-2023/)
- [Development Math PDF](https://ternercenter.berkeley.edu/wp-content/uploads/2023/12/Development-Math-2023.pdf)
- [Interactive Calculator](https://www.ternercenter.app/demystifying-development-math)
- [Assembly Housing Finance Hearing](https://committees.assembly.ca.gov/system/files/making-it-pencil-housing-finance-hearing.pdf)

In [ ]:
# Export scenario results
output_path = Path.cwd().parent / 'data' / 'outputs' / 'feasibility_scenarios.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(output_path, index=False)
print(f"Results exported to: {output_path}")